In [ ]:
import sys
import json
from pathlib import Path

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')

from load_data.load_conformal_data import load_conformance_results
from src.conformance_analysis.risk_model import DataFrameConstruction, LogisticRegressionModel
from evaluation.risk_classification_evaluation_metrics import risk_model_eval, risk_model_eval_median_fitness
from src.conformance_deviation_prediction.deviation import DeviationPrediction

In [ ]:
# Load the conformance evaluation results

# Helpdesk
# path_conformance_results = "../../../../../data/Helpdesk/proact_conf_check_v2/online_prediction/"
# path_model = "../1_conformance_analysis/results/Helpdesk/crc_logistic_model.pkl"
# path_dev_thresh = "../1_conformance_analysis/results/Helpdesk/deviation_labels_thresholds.json"
# New here: Store pkl of deviation prediction results
# path_pred_deviations = "./results/Helpdesk/deviation_predictions.pkl"

# Sepsis
# path_conformance_results = "../../../../../data/Sepsis/proact_conf_check_v2/online_prediction/"
# path_model = "../1_conformance_analysis/results/Sepsis/crc_logistic_model.pkl"
# path_dev_thresh = "../1_conformance_analysis/results/Sepsis/deviation_labels_thresholds.json"
# New here: Store pkl of deviation prediction results
# path_pred_deviations = "./results/Sepsis/deviation_predictions.pkl"

# BPIC20
# path_conformance_results = "../../../../../data/BPIC20/proact_conf_check_v2/online_prediction/"
# path_model = "../1_conformance_analysis/results/BPIC20/crc_logistic_model.pkl"
# path_dev_thresh = "../1_conformance_analysis/results/BPIC20/deviation_labels_thresholds.json"
# New here: Store pkl of deviation prediction results
# path_pred_deviations = "./results/BPIC20/deviation_predictions.pkl"

# Repair
path_conformance_results = "../../../../../data/repair_shop/proact_conf_check_v2/online_prediction/"
path_model = "../1_conformance_analysis/results/Repair/crc_logistic_model.pkl"
path_dev_thresh = "../1_conformance_analysis/results/Repair/deviation_labels_thresholds.json"
# New here: Store pkl of deviation prediction results
path_pred_deviations = "./results/Repair/deviation_predictions.pkl"


In [ ]:
# All results of Prob. Suffix Pred. + Alignment based conformance checking:
results = load_conformance_results(path=path_conformance_results)

# Case id:
res_case_id = results['case_id']
# Target conformance
res_target_conf = results['target_conformance']
# Most likely conformance
res_ml_conf = results['ml_conformance']
# Samples conformance:
res_smpl_conf = results['samples_conformance']
res_smpl_con_fit = [[r['suffix_fitness'] for r in res] for res in res_smpl_conf]

In [ ]:
# Load the logistic regression model and sort the conformance check results into risk and safe cases:
lm = LogisticRegressionModel().load(path_model)
print("Risk values and fitness threshold, determined in analysis and stored in model for reference:", lm.risk_fitness_threshold)

# Create dataframe for risk classification:
dc = DataFrameConstruction(conformance_results=results)

# Add as threshold the determined q_risk fitness score that is stored in the logistic regression model, to generate the targets
df = dc.samples_to_dataframe(q_risk=lm.risk_fitness_threshold,
                             target_col='y_safe_case',
                             include_tail_features=True)

In [ ]:
# Predict and sort cases into risk and safe:
risks = {'case_id': [],
         'label': [],
         'target_conformance': [],
         'ml_conformance': [],
         'samples_conformance': []}

safes = {'case_id': [],
         'label': [],
         'target_conformance': [],
         'ml_conformance': [],
         'samples_conformance': []}

# For evaluation purpose only!
eval_total = {'case_id': [],
              'label': [],
              'target_conformance': [],
              'ml_conformance': [],
              'samples_conformance': []}

# Use the CRC threshold: If >= t -> safe, else risk.
labels, probs = lm.predict_with_threshold(X=df)
for i in range(len(labels)):
    if labels[i] == 0:
        risks['case_id'].append(res_case_id[i])
        risks['label'].append(labels[i])
        risks['target_conformance'].append(res_target_conf[i])
        risks['ml_conformance'].append(res_ml_conf[i])
        risks['samples_conformance'].append(res_smpl_conf[i])
        
        eval_total['case_id'].append(res_case_id[i])
        eval_total['label'].append(labels[i])
        eval_total['target_conformance'].append(res_target_conf[i])
        eval_total['ml_conformance'].append(res_ml_conf[i])
        eval_total['samples_conformance'].append(res_smpl_conf[i])  
    else:
        safes['case_id'].append(res_case_id[i])
        safes['label'].append(labels[i])
        safes['target_conformance'].append(res_target_conf[i])
        safes['ml_conformance'].append(res_ml_conf[i])
        safes['samples_conformance'].append(res_smpl_conf[i])
        
        eval_total['case_id'].append(res_case_id[i])
        eval_total['label'].append(labels[i])
        
        res_target_conf[i]['suffix_alignment'] = []
        res_target_conf[i]['suffix_fitness'] = 1.0
        eval_total['target_conformance'].append(res_target_conf[i])
        
        res_ml_conf[i]['suffix_alignment'] = []
        res_ml_conf[i]['suffix_fitness'] = 1.0
        eval_total['ml_conformance'].append(res_ml_conf[i])
        
        for j in range(len(res_smpl_conf[i])):
            res_smpl_conf[i][j]['suffix_alignment'] = []
            res_smpl_conf[i][j]['suffix_fitness'] = 1.0
        eval_total['samples_conformance'].append(res_smpl_conf[i])
                  
print("Number of predicted cases as risk:", len(risks['case_id']))
print("Number of safe cases:", len(safes['case_id']))

In [ ]:
# Logistic regression evaluation -> risk and safe set classification:
risk_model_eval(labels=labels, probs=probs, y=df['y_safe_case'])

risk_model_eval_median_fitness(samples_fitness=res_smpl_con_fit,
                               y=df['y_safe_case'],
                               fitness_threshold=lm.risk_fitness_threshold)

In [ ]:
# Aggregated deviations:
dp = DeviationPrediction(pred_conf_set=eval_total)

# Probabilistic deviations:
path = Path(path_dev_thresh)
with path.open("r") as f:
    items = json.load(f)
thresholds = {tuple(d["key"]): d["value"] for d in items}

# deviation_results_evaluation = dp_evaluation.get_aggregated_deviations()
deviation_results_evaluation = dp.get_probabilistic_deviations_with_positions(deviation_thresholds=thresholds,
                                                                              eval_purpose=True)

dp.save(path=path_pred_deviations, deviations=deviation_results_evaluation)